<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

# **Trabajo Práctico 2: Reconstrucción 3D**

### **Opcional: normales y mallado con Poisson**


</div>

Este notebook parte de la nube final optimizada que genera `tp2.ipynb` (`outputs/stitch_final.ply`) y **no la
modifica**: estima las normales de la superficie y reconstruye una malla triangular con Poisson. Como solo lee el
`.ply`, no hace falta volver a correr el pipeline completo y los resultados del informe no cambian.

Igual que `tp2.ipynb`, hay que abrirlo con `notebooks/` como directorio de trabajo (importa `utils.py`).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Carga de la nube final**
</div>


Leemos la nube exportada por `tp2.ipynb`. Está en milímetros y en el marco del tablero ChArUco: x e y sobre el
tablero, z normal a él y hacia adentro (el peluche tiene z < 0).

In [ ]:
# Rutas del proyecto (el notebook corre desde notebooks/)
BASE = "." if os.path.isdir("datasets") else ".."

ply_path = os.path.join(BASE, "outputs", "stitch_final.ply")
pcd = o3d.io.read_point_cloud(ply_path)

P = np.asarray(pcd.points)
print(f"{ply_path}: {len(P)} puntos, con color: {pcd.has_colors()}, con normales: {pcd.has_normals()}")
print(f"Extensión (x, y, z): {np.round(P.max(axis=0) - P.min(axis=0), 1)} mm")

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Estimación de normales**
</div>


La normal en cada punto se estima con los vecinos: se ajusta un plano por PCA (la normal es el autovector de menor
autovalor de la covarianza local). Usamos un radio de 10 mm (5 vóxeles de 2 mm) y hasta 50 vecinos: con un radio
más chico las normales salen ruidosas por el pelaje y por el espesor que deja la fusión de vistas.

El PCA da la dirección pero no el sentido de la normal. Poisson necesita normales **orientadas de forma consistente**
(todas hacia afuera), así que usamos `orient_normals_consistent_tangent_plane`, que propaga el sentido entre vecinos
sobre un grafo. Después verificamos el sentido global: si la mayoría de las normales apunta hacia el centroide, las
damos vuelta.

In [ ]:
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=50))
pcd.orient_normals_consistent_tangent_plane(20)

# Sentido global: hacia afuera del objeto
N = np.asarray(pcd.normals)
afuera = np.mean(np.sum((P - P.mean(axis=0)) * N, axis=1) > 0)
if afuera < 0.5:
    pcd.normals = o3d.utility.Vector3dVector(-N)
    afuera = 1 - afuera

print(f"Normales hacia afuera del centroide: {afuera:.0%}")

No da 100 % porque el peluche no es convexo: en las orejas, los brazos y entre las piernas hay superficie bien
orientada cuya normal no se aleja del centroide.

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Mallado con Poisson**
</div>


La reconstrucción de Poisson busca una función indicadora χ (1 adentro del objeto, 0 afuera) cuyo gradiente se
parezca lo más posible al campo de normales orientadas, lo que lleva a resolver la ecuación de Poisson Δχ = ∇·**V**
sobre un octree. La superficie es una isosuperficie de χ.

1. **Poisson** con `depth=8`: el octree divide el cubo que encierra la nube en 2⁸ = 256 celdas por lado, unas
   1,5 mm, del orden del vóxel de 2 mm de la nube.
2. **Recorte por densidad**: Poisson siempre devuelve una superficie cerrada, así que "inventa" superficie donde no
   hay datos. Por ejemplo, la base quedó abierta al sacar el plano del tablero y Poisson la cierra con una "pollera"
   que atraviesa el tablero. `densities` indica cuántos puntos sostienen cada vértice; descartamos el 5 % menos
   sostenido, lo que elimina casi toda esa superficie inventada.
3. **Recorte con la caja de la nube**: descarta lo poco que queda fuera de la caja que encierra la nube.
4. **Suavizado de Taubin** (10 iteraciones): atenúa el ruido de pocos milímetros sin encoger la malla, a diferencia
   del suavizado laplaciano.
5. **Decimación** a 150 000 triángulos (`simplify_quadric_decimation`), para que el `.ply` quede liviano.

In [ ]:
steps = []

# 1. Poisson
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)
steps.append(("1. Poisson", mesh))

# 2. Recorte por densidad
densities = np.asarray(densities)
mesh = o3d.geometry.TriangleMesh(mesh)
mesh.remove_vertices_by_mask(densities < np.quantile(densities, 0.05))
steps.append(("2. recorte por densidad", mesh))

# 3. Recorte con la caja de la nube
mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())
steps.append(("3. recorte con la caja", mesh))

# 4. Suavizado de Taubin
mesh = mesh.filter_smooth_taubin(number_of_iterations=10)
steps.append(("4. Taubin", mesh))

# 5. Decimación
mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=150_000)
mesh.remove_unreferenced_vertices()
mesh.compute_vertex_normals()
mesh.vertex_colors = o3d.utility.Vector3dVector(np.clip(np.asarray(mesh.vertex_colors), 0, 1))
steps.append(("5. decimación (final)", mesh))

for name, m in steps:
    print(f"{name}: {len(m.vertices)} vértices, {len(m.triangles)} triángulos")

Para ver qué tan fiel es la malla a los datos medimos la distancia de cada vértice al punto más cercano de la nube.

In [ ]:
d = np.asarray(o3d.geometry.PointCloud(mesh.vertices).compute_point_cloud_distance(pcd))
print(f"Distancia vértice-nube: mediana {np.median(d):.2f} mm, percentil 95 {np.percentile(d, 95):.2f} mm, "
      f"máximo {d.max():.2f} mm")

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Visualización de la malla**
</div>


Mostramos la malla final desde el frente, el costado y la espalda (el peluche mira hacia −x del tablero), con el
color de la nube (arriba) y en gris con sombreado (abajo), que deja ver la geometría.

In [ ]:
from utils import plot_mesh

# Rotaciones mundo -> vista: (x de pantalla, y de pantalla, profundidad). y de pantalla = z del tablero.
views = [
    ("frente", np.array([[0, -1, 0], [0, 0, 1], [1, 0, 0]], float)),
    ("costado", np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0]], float)),
    ("espalda", np.array([[0, 1, 0], [0, 0, 1], [-1, 0, 0]], float)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for col, (name, R_view) in enumerate(views):
    plot_mesh(axes[0, col], mesh, R_view)
    plot_mesh(axes[1, col], mesh, R_view, colored=False)
    axes[0, col].set_title(name)

plt.tight_layout()
fig.savefig(os.path.join(BASE, "informe", "img", "poisson.png"), dpi=150, bbox_inches="tight")
plt.show()

La malla recupera la forma completa del peluche, incluida la espalda. La superficie se ve rugosa en gris: Poisson
reproduce el ruido de la nube (el pelaje y el leve engrosamiento por la fusión de vistas), que el suavizado atenúa
pero no elimina. El hueco entre el brazo y el cuerpo, que ninguna cámara observó con claridad, sigue abierto porque
el recorte por densidad descarta la superficie que Poisson inventa ahí.

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Exportación**
</div>


Guardamos la malla en `.ply` binario, aparte de la nube entregable. Se puede abrir en MeshLab, donde las normales
permiten verla con iluminación.

In [ ]:
mesh_path = os.path.join(BASE, "outputs", "stitch_poisson.ply")
o3d.io.write_triangle_mesh(mesh_path, mesh, write_ascii=False)

print(f"{mesh_path}: {os.path.getsize(mesh_path) / 1e6:.1f} MB, {len(mesh.vertices)} vértices, "
      f"{len(mesh.triangles)} triángulos")

<div style="text-align: center; font-family: 'Helvetica', sans-serif;">

### **Visualización interactiva en Open3D (opcional)**
</div>


Abre ventanas de Open3D con la nube y sus normales (tecla `N` para mostrarlas) y con la malla. No corre en Colab ni
sin pantalla: dejamos `SHOW_3D = False` para ejecutar el notebook de punta a punta.

In [ ]:
SHOW_3D = False

if SHOW_3D:
    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50)
    o3d.visualization.draw_geometries([pcd, frame], window_name="nube con normales", point_show_normal=True)
    o3d.visualization.draw_geometries([mesh, frame], window_name="malla de Poisson", mesh_show_back_face=True)